In [4]:
#import des données :
%pip install -r requirements.txt
import pandas as pd
import matplotlib.pyplot as plt
from cartiflette import carti_download
import seaborn as sns
import geopandas as gpd

url = "https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb"
df = pd.read_csv(
    url, 
    dtype={
        'code_departement': str, 
        'code_commune': str } )


Note: you may need to restart the kernel to use updated packages.


/tmp/ipykernel_1159/548435642.py:6: DtypeWarning: Columns (0: prenom) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [ ]:
#Question4)
#On ne laisse que les votes exprimés
exclusions = ['abstentions', 'blancs', 'nuls']
df_exprimes = df[~df['nom'].isin(exclusions)].copy()
#On crée candidat
df_exprimes['candidat'] = df_exprimes['prenom'] + ' ' + df_exprimes['nom']
#On regroupe
score_departements = df_exprimes.groupby(['code_departement', 'candidat']).agg({
    'voix': 'sum'
}).reset_index()
#On rajoute les %
total_par_dept = score_departements.groupby('code_departement')['voix'].transform('sum')
score_departements['score'] = (score_departements['voix'] / total_par_dept) * 100

In [ ]:
#Vérification TABLE 2

verif_table2=score_departements[score_departements['code_departement'] == '11'].sort_values(by='voix', ascending=False)
print(verif_table2)

In [ ]:
#Question5
#On crée les stats nationales
total_votes_national = score_departements['voix'].sum()
stats_national = score_departements.groupby('candidat').agg({
    'voix': 'sum'
}).reset_index()

stats_national.columns = ['candidat', 'votes_national']

stats_national['score_national'] = (stats_national['votes_national'] / total_votes_national) * 100
score_departements = score_departements.rename(columns={
    'voix': 'votes_departement',
    'score': 'score_departement'
})

#On fusionne
score_departements = pd.merge(score_departements, stats_national, on='candidat')



In [ ]:
#Vérification table3
verif_table_3 = score_departements[score_departements['code_departement'] == '11'].sort_values(by='votes_departement', ascending=False)
print(verif_table_3)

In [ ]:
# Question 6 

# Calcul de la surreprésentation relative (en %)
score_departements['surrepresentation'] = (
    (score_departements['score_departement'] / score_departements['score_national']) - 1
) * 100



In [ ]:
#Question 7 

def plot_top_5(df, nom_candidat):
    data = df[df['candidat'] == nom_candidat].sort_values('surrepresentation', ascending=False).head(5)
    plt.figure(figsize=(8, 5))
    sns.barplot(x='surrepresentation', y='code_departement', data=data, palette="Reds_r")
    plt.title(f"Top 5 des surreprésentations de {nom_candidat}")
    plt.xlabel("Surreprésentation (%)")
    plt.show()
    
plot_top_5(score_departements, "Éric ZEMMOUR")

In [ ]:
#Question 8 

def tracer_carte_candidat(nom_candidat):
    
    df_visu = score_departements[score_departements['candidat'] == nom_candidat].copy()
    
    # Jointure avec le fond de carte 'departement_borders'
    carte_data = departement_borders.merge(
        df_visu, 
        left_on='INSEE_DEP', 
        right_on='code_departement'
    )
    
    # Affichage de la carte
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    
    # On utilise 'surrepresentation' calculée à la Q6
    carte_data.plot(
        column='surrepresentation', 
        cmap='coolwarm', 
        legend=True, 
        ax=ax,
        center=0, # Important pour que le blanc soit à 0 (moyenne nationale)
        legend_kwds={'label': "% d'écart à la moyenne nationale"}
    )
    
    ax.set_title(f"Représentation territoriale de {nom_candidat} (Présidentielle 2022)")
    ax.axis('off') # Pour enlever les coordonnées GPS
    plt.show()

# Carte pour Marine Le Pen : 
tracer_carte_candidat("Marine LE PEN")